In [2]:
import polars as pl

df = pl.read_csv(
    "../data/lehner_dataset.txt",
    separator="\t",
    null_values="NA",  # Na interpeted as null
    infer_schema_length=10000
)

df

domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
str,str,str,str,i64,str,bool,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""*IFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""*""",true,118,113,62,10,29,2,97.66667,0.030945,0.014885,-0.81905,0.208478,339
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""AIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""A""",false,219,277,217,86,225,137,237.6667,0.069376,0.006673,-0.28079,0.093461,339
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""CIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""C""",false,706,726,459,768,507,616,630.3333,0.082052,0.004141,-0.10325,0.057995,339
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""DIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""D""",false,407,431,323,508,159,111,387.0,0.071003,0.005162,-0.258003,0.072296,339
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""EIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""E""",false,37,56,37,201,102,95,43.33333,0.1163261,0.012085,0.376783,0.169263,339
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q9Y6V0_PF05715_1058""","""Q9Y6V0""","""TWPLCKTELNIGSKDPPNFNTCTECKNQVC…","""C""",1059,"""W""",false,118,121,83,13,47,4,107.3333,0.028085,0.013317,-0.903822,0.15048,380
"""Q9Y6V0_PF05715_1058""","""Q9Y6V0""","""TYPLCKTELNIGSKDPPNFNTCTECKNQVC…","""C""",1059,"""Y""",false,177,168,126,12,6,29,157.0,0.006343,0.013753,-1.149495,0.155403,380
"""Q9Y6V0_PF05715_1058""","""Q9Y6V0""","""VCPLCKTELNIGSKDPPNFNTCTECKNQVC…","""T""",1058,"""V""",false,20,19,23,46,43,8,20.66667,0.09024,0.021806,-0.201507,0.246396,380


1. **`domain_ID`** – identifikátor domény proteinu, např. konkrétní segment v rámci celého proteinu.
2. **`uniprot_ID`** – unikátní ID proteinu podle databáze UniProt.
3. **`aa_seq`** – sekvence aminokyselin daného proteinu/domény.
4. **`wt_aa`** – „wild-type“ aminokyselina, tedy originální aminokyselina na dané pozici před mutací.
5. **`position`** – pozice aminokyseliny v sekvenci (indexace obvykle začíná od 1).
6. **`mut_aa`** – aminokyselina, která byla zavedena místo wild-type (mutovaná).
7. **`STOP`** – značí, zda mutace vede k stop-kodon (předčasnému ukončení proteinu).

---

8.–10. **`input_count_rep1`, `input_count_rep2`, `input_count_rep3`** – počty přečtení (read counts) mutace ve vstupní knihovně pro tři replikáty experimentu.
11.–13. **`output_count_rep1`, `output_count_rep2`, `output_count_rep3`** – počty přečtení mutace po selekci, tzn. kolik přežilo/aktivně fungovalo.

---

14. **`mean_input_count`** – průměr počtu vstupů přes replikáty, užitečné pro normalizaci.
15. **`fitness`** – odhad „fitness“ mutace, např. jak moc mutace ovlivňuje funkci proteinu (často log ratio output/input) (můžeš podle ní vážit loss funkci → čím menší sigma, tím víc věříš danému datu.).
16. **`fitness_sigma`** – chyba nebo odchylka od fitness měření (standard deviation).
17. **`normalized_fitness`** – fitness upravená pro porovnání mezi různými mutacemi/proteiny.
18. **`normalized_fitness_sigma`** – odchylka normalizované fitness (můžeš podle ní vážit loss funkci → čím menší sigma, tím víc věříš danému datu..
19. **`quality_rank`** – hodnocení kvality datové řádky, např. A/B/C podle spolehlivosti měření.

---

Jednoduše řečeno: **každý řádek odpovídá jedné mutaci** proteinu/domény a obsahuje informace o pozici, typu mutace a její experimentálně měřenou účinnost/stabilitu.



In [2]:
df["normalized_fitness"].describe()

statistic,value
str,f64
"""count""",563534.0
"""null_count""",39348.0
"""mean""",-0.318427
"""std""",0.385715
"""min""",-5.297542
"""25%""",-0.572715
"""50%""",-0.209113
"""75%""",-0.038835
"""max""",2.05257


In [3]:
import polars as pl
import requests
from io import StringIO
from tqdm import tqdm  # For progress bars


def fetch_uniprot_sequence(uniprot_id: str):
    """Fetch the original protein sequence from UniProt"""
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        if uniprot_id == "A2R8Y422":
            print(response.text)

        # Parse the FASTA file to extract the sequence
        fasta_data = StringIO(response.text)
        fasta_data.readline()  # skip header
        sequence = "".join(line.strip() for line in fasta_data)
        return sequence
    except requests.exceptions.RequestException as e:
        print(f"\nError fetching sequence for {uniprot_id}: {str(e)}")
        return None


df_with_parent = df

# Get unique UniProt IDs
uniprot_ids = df_with_parent["uniprot_ID"].unique().to_list()
print(f"Found {len(uniprot_ids)} unique UniProt IDs to process")

# Create a dictionary to store sequences
sequences = {}
failed_ids = []

# Fetch sequences for each UniProt ID with progress bar
print("\nDownloading sequences from UniProt:")
for uniprot_id in tqdm(uniprot_ids, desc="Progress"):
    if uniprot_id is None:
        continue
    sequence = fetch_uniprot_sequence(uniprot_id)

    if sequence or sequences == "":
        sequences[uniprot_id] = sequence
    else:
        sequences[uniprot_id] = None
        failed_ids.append(uniprot_id)

print("\nProcessing complete!")
print(f"Successfully retrieved {len(sequences)} sequences")
print(f"Failed to retrieve {len(failed_ids)} sequences")
if failed_ids:
    print("Failed IDs:", ", ".join(failed_ids))

# save to file mapping
import json

with open("uniprot_id_to_sequence_mapping.json", "w") as f:
    json.dump(sequences, f, indent=4)


Found 434 unique UniProt IDs to process



Progress:   4%|▍         | 18/434 [00:04<01:37,  4.26it/s]


KeyboardInterrupt: 

In [4]:
# adding missing

# load sequences from file
import json

sequences = {}
with open("uniprot_id_to_sequence_mapping.json", "r") as f:
    sequences = json.load(f)

mapping = {
    "P61960": "MSKVSFKITLTSDPRLPYKVLSVPESTPFTAVLKFAAEEFKVPAATSAIITNDGIGINPAQTAGNVFLKHGSELRIIPRDRVGSC",
    # https://www.uniprot.org/uniparc/UPI00000041DB/entry
    "Q3SY89": "MAAGSTTLRAVGKLQVRLATKTEPKKLEKYLQKLSALPMTADILAETGIRKTVKRLRKHQHVGDFARDLAARWKKLVLVDRNTGPDPQDPEESASRQRFGEALQEREKAWGFPENATAPRSPSHSPEHRRTARRTPPGQQRPHPRSPSREPRAERKRPRMAPADSGPHRDPPTRTAPLPMPEGPEPAVPGEQPGRGHAHAAQGGPLLGQGCQGQPQGEAVGSHSKGHKSSRGASAQKSPPVQESQSERLQAAGADSAGPKTVPSHVFSELWDPSEAWMQANYDLLSAFEAMTSQANPEALSAPTLQEEAAFPGRRVNAKMPVYSGSRPACQLQVPTLRQQCLRVPRNNPDALGDVEGVPYSVLEPVLEGWTPDQPYRTEKDNAALARETDELWRIHCLQDFKEEKPQEHESWRELYLRLRDAREQRLRVVTTKIRSARENKPSGRQTKMICFNSVAKTPYDASRRQEKSAGAADPGNGEMEPAPKPAGSSQAPSGLGDGDGGSVSGGGSSNRHAAPADKTRKQAAKKVAPLMAKAIRDYKGRFSRR",
    # https://www.uniprot.org/uniparc/UPI0000366CBD/entry
    "A0A2R8Y422": "MQIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGAKKRKKKSYTTPRKNKHKRKKVKLALLKYYKVDENGKISCLHRECPSDECGAGVFMASHFDRHYCGKCCLTYCFNKPEDK",
    # https://www.uniprot.org/uniparc/UPI000006FE7A/entry
    "P61086": "MANIAVQRIKREFKEVLKSEETSKNQIKVDLVDENFTELRGEIAGPPDTPYEGGRYQLEIKIPETYPFNPPKVRFITKIWHPNISSVTGAICLDILKDQWAAAMTLRTVLLSLQALLAAAEPDDPQDAVVANQYKQNPEMFKQTARLWAHVYAGAPVSSPEYTKKIENLCAMGFDRNAVIVALSSKSWDVETATELLLSN",
    #https://www.uniprot.org/uniparc/UPI0000003FF1/entry
    "A1X283": "MPPRRSIVEVKVLDVQKRRVPNKHYVYIIRVTWSSGSTEAIYRRYSKFFDLQMQMLDKFPMEGGQKDPKQRIIPFLPGKILFRRSHIRDVAVKRLIPIDEYCKALIQLPPYISQCDEVLQFFETRPEDLNPPKEEHIGKKKSGGDQTSVDPMVLEQYVVVANYQKQESSEISLSVGQVVDIIEKNESGWWFVSTAEEQGWVPATCLEGQDGVQDEFSLQPEEEEKYTVIYPYTARDQDEMNLERGAVVEVIQKNLEGWWKIRYQGKEGWAPASYLKKNSGEPLPPKPGPGSPSHPGALDLDGVSRQQNAVGREKELLSSQRDGRFEGRPVPDGDAKQRSPKMRQRPPPRRDMTIPRGLNLPKPPIPPQVEEEYYTIAEFQTTIPDGISFQAGLKVEVIEKNLSGWWYIQIEDKEGWAPATFIDKYKKTSNASRPNFLAPLPHEVTQLRLGEAAALENNTGSEATGPSRPLPDAPHGVMDSGLPWSKDWKGSKDVLRKASSDMSASAGYEEISDPDMEEKPSLPPRKESIIKSEGELLERERERQRTEQLRGPTPKPPGVILPMMPAKHIPPARDSRRPEPKPDKSRLFQLKNDMGLECGHKVLAKEVKKPNLRPISKSKTDLPEEKPDATPQNPFLKSRPQVRPKPAPSPKTEPPQGEDQVDICNLRSKLRPAKSQDKSLLDGEGPQAVGGQDVAFSRSFLPGEGPGRAQDRTGKQDGLSPKEISCRAPPRPAKTTDPVSKSVPVPLQEAPQQRPVVPPRRPPPPKKTSSSSRPLPEVRGPQCEGHESRAAPTPGRALLVPPKAKPFLSNSLGGQDDTRGKGSLGPWGTGKIGENREKAAAASVPNADGLKDSLYVAVADFEGDKDTSSFQEGTVFEVREKNSSGWWFCQVLSGAPSWEGWIPSNYLRKKP",
    # https://www.uniprot.org/uniparc/UPI000020C12E/entry

    "O95718": "MSSDDRHLGSSCGSFIKTEPSSPSSGIDALSHHSPSGSSDASGGFGLALGTHANGLDSPPMFAGAGLGGTPCRKSYEDCASGIMEDSAIKCEYMLNAIPKRLCLVCGDIASGYHYGVASCEACKAFFKRTIQGNIEYSCPATNECEITKRRRKSCQACRFMKCLKVGMLKEGVRLDRVRGGRQKYKRRLDSESSPYLSLQISPPAKKPLTKIVSYLLVAEPDKLYAMPPPGMPEGDIKALTTLCDLADRELVVIIGWAKHIPGFSSLSLGDQMSLLQSAWMEILILGIVYRSLPYDDKLVYAEDYIMDEEHSRLAGLLELYRAILQLVRRYKKLKVEKEEFVTLKALALANSDSMYIEDLEAVQKLQDLLHEALQDYELSQRHEEPWRTGKLLLTLPLLRQTAAKAVQHFYSVKLQGKVPMHKLFLEMLEAKAWARADSLQEWRPLEQVPSPLHRATKRQHVHFLTPLPPPPSVAWVGTAQAGYHLEVFLPQRAGWPRAA"
    # https://www.uniprot.org/uniparc/UPI000012A17F/entry

}

sequences.update(mapping)

df_with_parent = df.with_columns(
    pl.col("uniprot_ID").replace_strict(sequences, default=None).alias("original_sequence")
)

# check if all sequences are fetched
not_found = (
    df_with_parent.filter(pl.col("original_sequence").is_null())
    .select("uniprot_ID")
    .unique()
    .to_series()
    .to_list()
)

print(f"IDs not found in mapping: {not_found} with len {len(not_found)} ")

df_with_parent.write_csv("lehner_dataset_with_sequences.csv")

df_with_parent


IDs not found in mapping: ['HHH-rd1-0142', 'EEHEE-rd3-0037', 'O15350', 'Q01543', 'EHEE-rd1-0882'] with len 5 


domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank,original_sequence
str,str,str,str,i64,str,bool,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64,str
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""*IFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""*""",true,118,113,62,10,29,2,97.66667,0.030945,0.014885,-0.81905,0.208478,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…"
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""AIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""A""",false,219,277,217,86,225,137,237.6667,0.069376,0.006673,-0.28079,0.093461,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…"
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""CIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""C""",false,706,726,459,768,507,616,630.3333,0.082052,0.004141,-0.10325,0.057995,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…"
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""DIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""D""",false,407,431,323,508,159,111,387.0,0.071003,0.005162,-0.258003,0.072296,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…"
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""EIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""Q""",2,"""E""",false,37,56,37,201,102,95,43.33333,0.1163261,0.012085,0.376783,0.169263,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q9Y6V0_PF05715_1058""","""Q9Y6V0""","""TWPLCKTELNIGSKDPPNFNTCTECKNQVC…","""C""",1059,"""W""",false,118,121,83,13,47,4,107.3333,0.028085,0.013317,-0.903822,0.15048,380,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…"
"""Q9Y6V0_PF05715_1058""","""Q9Y6V0""","""TYPLCKTELNIGSKDPPNFNTCTECKNQVC…","""C""",1059,"""Y""",false,177,168,126,12,6,29,157.0,0.006343,0.013753,-1.149495,0.155403,380,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…"
"""Q9Y6V0_PF05715_1058""","""Q9Y6V0""","""VCPLCKTELNIGSKDPPNFNTCTECKNQVC…","""T""",1058,"""V""",false,20,19,23,46,43,8,20.66667,0.09024,0.021806,-0.201507,0.246396,380,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…"


# Normalization of fitness using piecewise sigmoid function

In [5]:
import polars as pl
import numpy as np

df_with_parent = pl.read_csv(
    "lehner_dataset_with_sequences.csv",
    infer_schema_length=10000
)

# creating reverse dataset_mutation

# original seq are not reverse
df_with_parent = df_with_parent.with_columns(pl.lit(False).alias("reverse"))

# Reverzní mutace
df_with_parent_with_reverse = df_with_parent.with_columns([
    pl.col("mut_aa").alias("wt_aa"),
    pl.col("wt_aa").alias("mut_aa"),
    # multiply by -1
    pl.col("normalized_fitness") * -1,

    pl.lit(True).alias("reverse")
])

df_with_parent = pl.concat([df_with_parent_with_reverse, df_with_parent])

k_neg = 0.871
k_pos = 0.871
A_neg = 1.0
A_pos = 1.0

sigmoid_expr = (
    pl.when(pl.col("normalized_fitness") >= 0)
    .then(A_pos * (2 / (1 + (-k_pos * pl.col("normalized_fitness")).exp()) - 1))
    .otherwise(-A_neg * (2 / (1 + (-k_neg * (-pl.col("normalized_fitness"))).exp()) - 1))
)

df_with_parent_normalized = df_with_parent.with_columns(
    sigmoid_expr.alias("normalized_fitness_sigmoid")
)

df_with_parent_normalized["normalized_fitness_sigmoid"].describe()

df_with_parent_normalized


domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank,original_sequence,reverse,normalized_fitness_sigmoid
str,str,str,str,i64,str,bool,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64,str,bool,f64
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""*IFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""*""",2,"""Q""",true,118,113,62,10,29,2,97.66667,0.030945,0.014885,0.81905,0.208478,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,0.342301
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""AIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""A""",2,"""Q""",false,219,277,217,86,225,137,237.6667,0.069376,0.006673,0.28079,0.093461,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,0.121678
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""CIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""C""",2,"""Q""",false,706,726,459,768,507,616,630.3333,0.082052,0.004141,0.10325,0.057995,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,0.044935
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""DIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""D""",2,"""Q""",false,407,431,323,508,159,111,387.0,0.071003,0.005162,0.258003,0.072296,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,0.11189
"""A0A2R8Y422_PF00240_2""","""A0A2R8Y422""","""EIFVKTLMGKTITLEVELSDTIDNVKAKIQ…","""E""",2,"""Q""",false,37,56,37,201,102,95,43.33333,0.1163261,0.012085,-0.376783,0.169263,339,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,-0.162632
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q9Y6V0_PF05715_1058""","""Q9Y6V0""","""TWPLCKTELNIGSKDPPNFNTCTECKNQVC…","""C""",1059,"""W""",false,118,121,83,13,47,4,107.3333,0.028085,0.013317,-0.903822,0.15048,380,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,-0.374472
"""Q9Y6V0_PF05715_1058""","""Q9Y6V0""","""TYPLCKTELNIGSKDPPNFNTCTECKNQVC…","""C""",1059,"""Y""",false,177,168,126,12,6,29,157.0,0.006343,0.013753,-1.149495,0.155403,380,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,-0.462593
"""Q9Y6V0_PF05715_1058""","""Q9Y6V0""","""VCPLCKTELNIGSKDPPNFNTCTECKNQVC…","""T""",1058,"""V""",false,20,19,23,46,43,8,20.66667,0.09024,0.021806,-0.201507,0.246396,380,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,-0.087532


In [6]:
df_output = df_with_parent_normalized.select(
    ["aa_seq", "normalized_fitness_sigmoid", "position", "original_sequence", "reverse", "mut_aa", "wt_aa"])

df_output = df_output.rename({"position": "local_position"})

df_output

aa_seq,normalized_fitness_sigmoid,local_position,original_sequence,reverse,mut_aa,wt_aa
str,f64,i64,str,bool,str,str
"""*IFVKTLMGKTITLEVELSDTIDNVKAKIQ…",0.342301,2,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,"""Q""","""*"""
"""AIFVKTLMGKTITLEVELSDTIDNVKAKIQ…",0.121678,2,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,"""Q""","""A"""
"""CIFVKTLMGKTITLEVELSDTIDNVKAKIQ…",0.044935,2,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,"""Q""","""C"""
"""DIFVKTLMGKTITLEVELSDTIDNVKAKIQ…",0.11189,2,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,"""Q""","""D"""
"""EIFVKTLMGKTITLEVELSDTIDNVKAKIQ…",-0.162632,2,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,"""Q""","""E"""
…,…,…,…,…,…,…
"""TWPLCKTELNIGSKDPPNFNTCTECKNQVC…",-0.374472,1059,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,"""W""","""C"""
"""TYPLCKTELNIGSKDPPNFNTCTECKNQVC…",-0.462593,1059,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,"""Y""","""C"""
"""VCPLCKTELNIGSKDPPNFNTCTECKNQVC…",-0.087532,1058,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,"""V""","""T"""


In [16]:
import re

df_output_tmp = (
    df_output.with_columns(
        # Step 3: Create the final required columns using the calculated global_position.
        mut_type=pl.concat_str(
            [pl.col("mut_aa"), (pl.col("local_position") ), pl.col("wt_aa")]
        ),

        mutated_seq_full=pl.concat_str(
            [
                pl.col("original_sequence").str.slice(0, pl.col("local_position") - 1),
                pl.col("mut_aa"),
                pl.col("original_sequence").str.slice(pl.col("local_position")),
            ]
        ),

        original_seq_full=pl.concat_str(
            [
                pl.col("original_sequence").str.slice(0, pl.col("local_position") - 1),
                pl.col("wt_aa"),
                pl.col("original_sequence").str.slice(pl.col("local_position")),
            ]
        ),
    )
    .select(
        # Select and rename columns for the final output
        pl.col("mut_type"),
        pl.col("original_seq_full"),
        pl.col("mutated_seq_full"),
        pl.col("normalized_fitness_sigmoid"),
        pl.col("reverse"),

    )
)
df_output_tmp = df_output_tmp.drop_nulls(["original_seq_full", "mutated_seq_full", "normalized_fitness_sigmoid"])

df_output_tmp.write_csv("datasets/lehner_dataset_with_sequences_normalized.csv")

df_output_tmp

mut_type,original_seq_full,mutated_seq_full,normalized_fitness_sigmoid,reverse
str,str,str,f64,bool
"""Q2*""","""M*IFVKTLMGKTITLEVELSDTIDNVKAKI…","""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.342301,true
"""Q2A""","""MAIFVKTLMGKTITLEVELSDTIDNVKAKI…","""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.121678,true
"""Q2C""","""MCIFVKTLMGKTITLEVELSDTIDNVKAKI…","""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.044935,true
"""Q2D""","""MDIFVKTLMGKTITLEVELSDTIDNVKAKI…","""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.11189,true
"""Q2E""","""MEIFVKTLMGKTITLEVELSDTIDNVKAKI…","""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",-0.162632,true
…,…,…,…,…
"""V1059C""","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",-0.45026,false
"""W1059C""","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",-0.374472,false
"""Y1059C""","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",-0.462593,false


In [17]:
df_output_tmp.filter(pl.col('original_seq_full') == pl.col('mutated_seq_full'))

mut_type,original_seq_full,mutated_seq_full,normalized_fitness_sigmoid,reverse
str,str,str,f64,bool
